<a href="https://colab.research.google.com/github/157-Fadina/REPO-KELOMPOK04-MAHADATA/blob/main/Kelompok04_Mahadata_M09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **KELOMPOK 04 - TUGAS BESAR - MAHADATA**

Nama Kelompok :
1. Fadina Mustika Ratnaningsih (NIM 123140157)
2. Firman H Gultom (NIM 123140171)
3. Atika Adelia (NIM 123140172)
4. Najwa Syahirah Rosyan (NIM 123140178)

In [1]:
!pip install -q pyspark kagglehub

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os, shutil, json, uuid, hashlib
from functools import reduce
from datetime import datetime
import pandas as pd
import kagglehub
from pyspark.sql import SparkSession, functions as F, types as T

In [4]:
# Lokasi Penyimpanan
# Google Drive
BASE      = "/content/drive/MyDrive/fraud_detection"
# Transit lokal (cepat, hilang saat runtime reset)
LOCAL_TMP = "/content/_tmp"
# File Data Asli
RAW   = f"{BASE}/00_raw/kaggle/creditcardfraud"
# Cluster MongoDB Simulasi
MDB   = f"{BASE}/01_mongodb_sim"
# Record yang gagal validasi
QUAR  = f"{BASE}/02_quarantine"
COMMIT_LOG = f"{MDB}/_meta/commit_log.json"

#  Identitas collection dan cluster MongoDB
DB, COL  = "fraud_db", "transactions"
NODES    = ["node_primary", "node_secondary_1", "node_secondary_2"]
# 3 node Ke 2
MAJORITY = len(NODES) // 2 + 1
SOURCE   = "kaggle:mlg-ulb/creditcardfraud"
BATCH_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
# area transit sebelum commit
STAGING  = f"{LOCAL_TMP}/staging/{BATCH_ID}"

for p in (RAW, MDB, QUAR, LOCAL_TMP):
    os.makedirs(p, exist_ok=True)
pd.set_option("display.max_columns", None)

In [5]:
# Checksum SHA-256 setiap part-file JSON dalam satu folder
def dir_checksum(path):
    out = {}
    for name in sorted(os.listdir(path)):
        if name.startswith("part-") and name.endswith(".json"):
            h = hashlib.sha256()
            with open(os.path.join(path, name), "rb") as f:
                for chunk in iter(lambda: f.read(1 << 20), b""):
                    h.update(chunk)
            out[name] = h.hexdigest()
    return out

print("BATCH_ID:", BATCH_ID, "| mayoritas:", MAJORITY, "dari", len(NODES), "node")

BATCH_ID: 20261028_163644_bf4c41 | mayoritas: 2 dari 3 node


In [6]:
spark = (SparkSession.builder
         .master("local[*]")
         .appName("FraudDetection-Load")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
print("Spark", spark.version, "| master:", spark.sparkContext.master)

Spark 4.0.4 | master: local[*]


In [7]:
CSV_PATH = f"{RAW}/creditcard.csv"
if not os.path.exists(CSV_PATH):
    src = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
    shutil.copy(os.path.join(src, "creditcard.csv"), CSV_PATH)
print(os.listdir(RAW), f"| {os.path.getsize(CSV_PATH)/1e6:.0f} MB")

Using Colab cache for faster access to the 'creditcardfraud' dataset.
['creditcard.csv'] | 151 MB


In [8]:
feature_cols = [f"V{i}" for i in range(1, 29)]
schema = T.StructType(
    [T.StructField("Time", T.DoubleType())] +
    [T.StructField(c, T.DoubleType()) for c in feature_cols] +
    [T.StructField("Amount", T.DoubleType()),
     T.StructField("Class", T.IntegerType())])

# Skema dokumen di collection
doc_schema = T.StructType(
    [T.StructField("_id", T.StringType())] + list(schema.fields) +
    [T.StructField("_ingested_at", T.TimestampType()),
     T.StructField("_batch_id", T.StringType()),
     T.StructField("_source", T.StringType())])

df = spark.read.csv(CSV_PATH, header=True, schema=schema).cache()
raw_count = df.count()
print(f"Baris mentah: {raw_count:,} | Kolom: {len(df.columns)}")

# Tabel data mentah
df.limit(10).toPandas()

Baris mentah: 284,807 | Kolom: 31


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
5,2.0,-0.425966,0.960523,1.141109,-0.168252,0.420987,-0.029728,0.476201,0.260314,-0.568671,-0.371407,1.341262,0.359894,-0.358091,-0.137134,0.517617,0.401726,-0.058133,0.068653,-0.033194,0.084968,-0.208254,-0.559825,-0.026398,-0.371427,-0.232794,0.105915,0.253844,0.081080,3.67,0
6,4.0,1.229658,0.141004,0.045371,1.202613,0.191881,0.272708,-0.005159,0.081213,0.464960,-0.099254,-1.416907,-0.153826,-0.751063,0.167372,0.050144,-0.443587,0.002821,-0.611987,-0.045575,-0.219633,-0.167716,-0.270710,-0.154104,-0.780055,0.750137,-0.257237,0.034507,0.005168,4.99,0
7,7.0,-0.644269,1.417964,1.074380,-0.492199,0.948934,0.428118,1.120631,-3.807864,0.615375,1.249376,-0.619468,0.291474,1.757964,-1.323865,0.686133,-0.076127,-1.222127,-0.358222,0.324505,-0.156742,1.943465,-1.015455,0.057504,-0.649709,-0.415267,-0.051634,-1.206921,-1.085339,40.80,0
8,7.0,-0.894286,0.286157,-0.113192,-0.271526,2.669599,3.721818,0.370145,0.851084,-0.392048,-0.410430,-0.705117,-0.110452,-0.286254,0.074355,-0.328783,-0.210077,-0.499768,0.118765,0.570328,0.052736,-0.073425,-0.268092,-0.204233,1.011592,0.373205,-0.384157,0.011747,0.142404,93.20,0
9,9.0,-0.338262,1.119593,1.044367,-0.222187,0.499361,-0.246761,0.651583,0.069539,-0.736727,-0.366846,1.017614,0.836390,1.006844,-0.443523,0.150219,0.739453,-0.540980,0.476677,0.451773,0.203711,-0.246914,-0.633753,-0.120794,-0.385050,-0.069733,0.094199,0.246219,0.083076,3.68,0


In [9]:
# NULL dianggap tidak valid
is_valid = F.coalesce(
    reduce(lambda a, b: a & b, [F.col(c).isNotNull() for c in feature_cols]) &
    F.col("Time").isNotNull() & (F.col("Time") >= 0) &
    F.col("Amount").isNotNull() & (F.col("Amount") >= 0) &
    F.col("Class").isin(0, 1),
    F.lit(False))

df_ok, df_bad = df.filter(is_valid), df.filter(~is_valid)
bad_count = df_bad.count()
# disimpan untuk audit
if bad_count > 0:
    tmp = f"{LOCAL_TMP}/quarantine/{BATCH_ID}"
    df_bad.write.mode("errorifexists").json(tmp)
    shutil.copytree(tmp, f"{QUAR}/{BATCH_ID}")
ok_count = raw_count - bad_count
print(f"Lolos validasi: {ok_count:,} | Quarantine: {bad_count:,}")

Lolos validasi: 284,807 | Quarantine: 0


In [10]:
doc = (df_ok
  .withColumn("_id", F.sha2(F.concat_ws("|", *[F.col(c).cast("string") for c in df_ok.columns]), 256))
  .dropDuplicates(["_id"])
  .withColumn("_ingested_at", F.current_timestamp())
  .withColumn("_batch_id", F.lit(BATCH_ID))
  .withColumn("_source", F.lit(SOURCE)))

# Staging belum terlihat oleh pembaca manapun (belum committed)
doc.coalesce(4).write.mode("errorifexists").json(STAGING)

staged = spark.read.schema(doc_schema).json(STAGING)
staged_rows = staged.count()
print(f"Baris mentah        : {raw_count:,}")
print(f"Ditolak (quarantine): {bad_count:,}")
print(f"Duplikat dibuang    : {ok_count - staged_rows:,}")
print(f"Dokumen di staging  : {staged_rows:,}")

staged.limit(5).toPandas()

Baris mentah        : 284,807
Ditolak (quarantine): 0
Duplikat dibuang    : 1,081
Dokumen di staging  : 283,726


,_id,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class,_ingested_at,_batch_id,_source
0,4c7d2f5ca10f100a9f758071f86640c00d34bcc3b54a28...,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
1,21b9b085ff4e22d083f29cc4fa8ab45519e0babf5d60a5...,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
2,ef4562ccb9d19866b5a551a02ac576fee692ce9e28aef4...,10.0,1.449044,-1.176339,0.913860,-1.375667,-1.971383,-0.629152,-1.423236,0.048456,-1.720408,1.626659,1.199644,-0.671440,-0.513947,-0.095045,0.230930,0.031967,0.253415,0.854344,-0.221365,-0.387226,-0.009302,0.313894,0.027740,0.500512,0.251367,-0.129478,0.042850,0.016253,7.80,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
3,e1d9dfed9fe624b8e148f377ed8f78fcd4031b239405ee...,18.0,1.166616,0.502120,-0.067300,2.261569,0.428804,0.089474,0.241147,0.138082,-0.989162,0.922175,0.744786,-0.531377,-2.105346,1.126870,0.003075,0.424425,-0.454475,-0.098871,-0.816597,-0.307169,0.018702,-0.061972,-0.103855,-0.370415,0.603200,0.108556,-0.040521,-0.011418,2.28,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
4,5362c3d5fcc09236a66818ee9f562eebceaa47073b147c...,22.0,-2.074295,-0.121482,1.322021,0.410008,0.295198,-0.959537,0.543985,-0.104627,0.475664,0.149451,-0.856566,-0.180523,-0.655233,-0.279797,-0.211668,-0.333321,0.010751,-0.488473,0.505751,-0.386694,-0.403639,-0.227404,0.742435,0.398535,0.249212,0.274404,0.359969,0.243232,26.43,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud


In [11]:
# Simulasi partisi/node mati
SIMULASI_NODE_GAGAL = []

expected_ck = dir_checksum(STAGING)
assert expected_ck, "Staging kosong: tidak ada part-file JSON"

committed = []
for node in NODES:
    target = f"{MDB}/{node}/{DB}/{COL}"
    if os.path.exists(target):
        print(f"[SKIP ] {node}: collection sudah ada")
        continue
    try:
        if node in SIMULASI_NODE_GAGAL:
            raise IOError("node tidak dapat dijangkau (simulasi partisi)")
        shutil.copytree(STAGING, target, ignore=shutil.ignore_patterns(".*.crc"))
        if dir_checksum(target) != expected_ck:
            raise ValueError("checksum berbeda dari staging")
        committed.append(node)
        print(f"[OK   ] {node}")
    except Exception as e:
        shutil.rmtree(target, ignore_errors=True)
        print(f"[GAGAL] {node}: {e}")

print(f"Node lolos verifikasi: {len(committed)}/{len(NODES)} (mayoritas = {MAJORITY})")

if len(committed) >= MAJORITY:
    # commit: catat di commit log; baru setelah ini data dianggap sah
    os.makedirs(os.path.dirname(COMMIT_LOG), exist_ok=True)
    entries = []
    if os.path.exists(COMMIT_LOG):
        with open(COMMIT_LOG) as f:
            entries = json.load(f)
    entries.append({"batch": BATCH_ID, "rows": staged_rows, "nodes": committed,
                    "source": SOURCE, "ts": datetime.now().isoformat(timespec="seconds"),
                    "checksums": expected_ck})
    with open(COMMIT_LOG + ".tmp", "w") as f:
        json.dump(entries, f, indent=2)
    os.replace(COMMIT_LOG + ".tmp", COMMIT_LOG)
    shutil.rmtree(STAGING)
    print(f"COMMIT berhasil: {staged_rows:,} dokumen di {len(committed)} node")
else:
    # rollback: konsistensi diprioritaskan di atas ketersediaan
    for node in committed:
        shutil.rmtree(f"{MDB}/{node}/{DB}/{COL}", ignore_errors=True)
    raise RuntimeError(f"Mayoritas node tidak tercapai ({len(committed)}/{len(NODES)}): write ditolak")

[OK   ] node_primary
[OK   ] node_secondary_1
[OK   ] node_secondary_2
Node lolos verifikasi: 3/3 (mayoritas = 2)
COMMIT berhasil: 283,726 dokumen di 3 node


In [12]:
with open(COMMIT_LOG) as f:
    last = json.load(f)[-1]
# batch committed terbaru
print("Batch committed:", {k: last[k] for k in ("batch", "rows", "nodes", "ts")})

read_node = last["nodes"][0]
path = f"{MDB}/{read_node}/{DB}/{COL}"
assert dir_checksum(path) == last["checksums"], f"Integritas {read_node} tidak cocok dengan commit log"

txn = spark.read.schema(doc_schema).json(path)
n = txn.count()
assert n == last["rows"], f"Jumlah dokumen {n} != commit log {last['rows']}"
print(f"Membaca dari {read_node}: {n:,} dokumen | {len(txn.columns)} kolom")

# tabel hasil akhir dari collection yang sudah committed
txn.limit(10).toPandas()

Batch committed: {'batch': '20261028_163644_bf4c41', 'rows': 283726, 'nodes': ['node_primary', 'node_secondary_1', 'node_secondary_2'], 'ts': '2026-10-28T16:38:36'}
Membaca dari node_primary: 283,726 dokumen | 35 kolom


,_id,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class,_ingested_at,_batch_id,_source
0,4c7d2f5ca10f100a9f758071f86640c00d34bcc3b54a28...,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
1,21b9b085ff4e22d083f29cc4fa8ab45519e0babf5d60a5...,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
2,ef4562ccb9d19866b5a551a02ac576fee692ce9e28aef4...,10.0,1.449044,-1.176339,0.913860,-1.375667,-1.971383,-0.629152,-1.423236,0.048456,-1.720408,1.626659,1.199644,-0.671440,-0.513947,-0.095045,0.230930,0.031967,0.253415,0.854344,-0.221365,-0.387226,-0.009302,0.313894,0.027740,0.500512,0.251367,-0.129478,0.042850,0.016253,7.80,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
3,e1d9dfed9fe624b8e148f377ed8f78fcd4031b239405ee...,18.0,1.166616,0.502120,-0.067300,2.261569,0.428804,0.089474,0.241147,0.138082,-0.989162,0.922175,0.744786,-0.531377,-2.105346,1.126870,0.003075,0.424425,-0.454475,-0.098871,-0.816597,-0.307169,0.018702,-0.061972,-0.103855,-0.370415,0.603200,0.108556,-0.040521,-0.011418,2.28,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
4,5362c3d5fcc09236a66818ee9f562eebceaa47073b147c...,22.0,-2.074295,-0.121482,1.322021,0.410008,0.295198,-0.959537,0.543985,-0.104627,0.475664,0.149451,-0.856566,-0.180523,-0.655233,-0.279797,-0.211668,-0.333321,0.010751,-0.488473,0.505751,-0.386694,-0.403639,-0.227404,0.742435,0.398535,0.249212,0.274404,0.359969,0.243232,26.43,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
5,50df0247764d0b410d56d61db0e4edc6d098191b96afc8...,35.0,1.386397,-0.794209,0.778224,-0.864708,-1.064132,0.351296,-1.191455,0.052686,-0.304404,0.576517,-1.631112,0.042560,2.047898,-0.739338,1.456222,-0.272050,-0.932007,1.926532,-0.659939,-0.273033,-0.228727,-0.123522,-0.131025,-0.929668,0.181379,1.194928,0.000531,0.019911,30.90,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
6,89d70585b00748303805070d72c595ae9aacfa79e87cfe...,41.0,1.154312,0.265462,0.384871,0.575007,-0.217475,-0.391520,-0.081489,0.062789,-0.260583,-0.161677,2.052928,1.130826,0.189485,-0.070793,0.619345,0.318583,0.090232,-0.331450,-0.344721,-0.099580,-0.193213,-0.557685,0.169492,0.186863,0.089252,0.093626,-0.009633,0.019668,2.67,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
7,c35e667ea505301e1e6251b1be2587100933ed5a9d669e...,71.0,1.331897,-0.579962,0.422606,-0.897752,-0.746254,-0.056273,-0.750317,0.128484,-0.964682,0.736391,1.390982,-0.008049,-0.129422,0.094277,0.388819,1.236619,0.151535,-1.398990,0.672575,0.052489,-0.039070,-0.255174,0.109333,-0.328448,0.162254,-0.447276,0.020071,0.006231,14.48,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
8,879513bef9c51785e996e48b3d552912c720b5f1aca51c...,73.0,1.148187,0.085837,0.120702,1.126665,0.214711,0.537381,-0.049989,0.186175,0.111781,0.046609,0.507647,0.869540,-0.322736,0.269322,-0.717472,-0.313732,-0.223905,-0.227786,0.179132,-0.148395,-0.091112,-0.095475,-0.166750,-0.653433,0.713020,-0.288035,0.031507,0.000372,19.77,0,2026-10-28 16:37:50.387,20261028_163644_bf4c41,kaggle:mlg-ulb/creditcardfraud
9,3077a4d320b956e40279dc9bdfb92b2f8c1f09bb715d84...,76.0,-1.024576,0.522289,1.787699,0.202672,-1.140803,-0.137831,-0.336555,0.670704,0.071670,-0.518978,0.687066,0.249020,-1.2486